# Extract Patches

> Extract single pins from the whole image

In [ ]:
#| default_exp extract_patches

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from pathlib import Path
from argparse import ArgumentParser
import shutil
from tqdm.auto import tqdm
from typing import List, Tuple, Union
from fastcore.script import *
import cv2

In [ ]:
#| export
import sys
sys.path.append(str(Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/new_test')))
sys.path.append(str(Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')))

In [ ]:
#| export
from cv_tools.core import *

In [ ]:
#| export
def process_file(args):
    """Process a single mask file to extract pins.
    
    Args:
        args: Tuple containing (mask_file, image_dir, output_base_dir, patch_size)
    """
    mask_file, image_dir, output_base_dir, patch_size = args
    # Get corresponding image file
    img_file = image_dir / mask_file.name
    if not img_file.exists():
        print(f"Warning: No matching image for mask {mask_file.name}")
        return
    
    # Create output directory for this image
    #output_dir = output_base_dir / mask_file.stem
    output_base_dir.mkdir(exist_ok=True, parents=True)
    
    # Extract pins
    extract_pins_from_mask(
        img_file, 
        mask_file, 
        output_base_dir, 
        patch_size, 
        False)

In [ ]:
#| export
def extract_pins_from_mask(
    image_path: str,
    mask_path: str,
    output_dir: str,
    patch_size: Tuple[int, int] = (87, 70),
    parallel: bool = True
) -> None:
    """
    Extract individual pin patches from an image based on a binary mask.
    
    Args:
        image_path: Path to the source image
        mask_path: Path to the binary mask
        output_dir: Directory to save extracted patches
        patch_size: Size of patches to extract (height, width)
        parallel: Whether to process in parallel
    """
    # Convert paths to Path objects
    image_path = Path(image_path)
    mask_path = Path(mask_path)
    output_dir = Path(output_dir)
    
    # Create output directories
    pin_img_dir = output_dir / "pin_images"
    pin_mask_dir = output_dir / "pin_masks"
    pin_img_dir.mkdir(exist_ok=True, parents=True)
    pin_mask_dir.mkdir(exist_ok=True, parents=True)
    
    # Read image and mask
    img = read_img(image_path, gray=False, cv=True)
    mask = read_img(mask_path, gray=True, cv=True)
    
    # Ensure mask is binary
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    
    # Find contours in the mask
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    def process_contour(idx, contour):
        # Get bounding box
        x, y, w, h = cv2.boundingRect(contour)
        
        # Calculate center of the contour
        center_x = x + w // 2
        center_y = y + h // 2
        
        # Calculate top-left corner for the patch
        patch_x = max(0, center_x - patch_size[1] // 2)
        patch_y = max(0, center_y - patch_size[0] // 2)
        
        # Extract patches
        img_patch = img[patch_y:patch_y+patch_size[0], patch_x:patch_x+patch_size[1]]
        mask_patch = binary_mask[patch_y:patch_y+patch_size[0], patch_x:patch_x+patch_size[1]]
        
        # Handle patches that might be smaller than the desired size due to image boundaries
        if img_patch.shape[0] < patch_size[0] or img_patch.shape[1] < patch_size[1]:
            # Create a blank canvas of the desired size
            img_canvas = np.zeros((patch_size[0], patch_size[1], 3), dtype=np.uint8)
            mask_canvas = np.zeros((patch_size[0], patch_size[1]), dtype=np.uint8)
            
            # Copy the patch onto the canvas
            h, w = img_patch.shape[:2]
            img_canvas[:h, :w] = img_patch
            mask_canvas[:h, :w] = mask_patch
            
            img_patch = img_canvas
            mask_patch = mask_canvas
        
        # Save patches
        cv2.imwrite(str(pin_img_dir / f"{Path(mask_path.stem)}_pin_{idx:04d}.png"), img_patch)
        cv2.imwrite(str(pin_mask_dir / f"{Path(mask_path.stem)}_pin_{idx:04d}.png"), mask_patch)
    
    # Process contours
    if parallel and len(contours) > 1:
        from concurrent.futures import ProcessPoolExecutor
        with ProcessPoolExecutor() as executor:
            list(tqdm(executor.map(process_contour, range(len(contours)), contours), 
                     total=len(contours), desc="Extracting pins"))
    else:
        for idx, contour in tqdm(enumerate(contours), total=len(contours), desc="Extracting pins"):
            process_contour(idx, contour)
    
    print(f"Extracted {len(contours)} pins to {output_dir}")


In [ ]:
#| export
@call_parse
def batch_extract_pins(
    image_dir: Param(
        "Directory containing source images", 
        type=str)='image_dir',
    mask_dir: Param(
        "Directory containing binary masks", type=str
        )='mask_dir',
    output_base_dir: Param("Base directory to save extracted patches", type=str)='output_base_dir',
    patch_size: Param("Size of patches to extract (height, width)", type=Tuple[int, int])=(87, 70),
    parallel: Param("Whether to process in parallel", action='store_true')=True
) -> None:
    """
    Process multiple images and masks to extract pin patches.
    
    Args:
        image_dir: Directory containing source images
        mask_dir: Directory containing binary masks
        output_base_dir: Base directory to save extracted patches
        patch_size: Size of patches to extract (height, width)
        parallel: Whether to process in parallel
    """
    image_dir = Path(image_dir)
    mask_dir = Path(mask_dir)
    output_base_dir = Path(output_base_dir)
    Path(output_base_dir).mkdir(exist_ok=True, parents=True)
    
    # Get all mask files
    mask_files = Path(mask_dir).ls(file_exts=['.png'])
    if not mask_files:
        mask_files = list(mask_dir.glob("*.jpg"))
    
    # Create arguments for process_file
    process_args = [(mask_file, image_dir, output_base_dir, patch_size) 
                   for mask_file in mask_files]
    
    # Process all files
    if parallel and len(mask_files) > 1:
        from concurrent.futures import ProcessPoolExecutor
        with ProcessPoolExecutor() as executor:
            list(tqdm(executor.map(process_file, process_args), 
                     total=len(mask_files), desc="Processing images"))
    else:
        for args in tqdm(process_args, desc="Processing images"):
            process_file(args)

> test

In [ ]:
from platform import system
if system() == 'Windows':
    image_dir =Path(r'E:\CurrentTrainingData20240812_trn_val\Incoming_1B_Loetstift\Incoming_1B_Loetstift_unzip\main_im2_cropped_masks\time_11_04_21_val_frGrnd0.9498_epoch_200.h5\failed\missing\images')
    mask_dir =Path(r'E:\CurrentTrainingData20240812_trn_val\Incoming_1B_Loetstift\Incoming_1B_Loetstift_unzip\main_im2_cropped_masks\time_11_04_21_val_frGrnd0.9498_epoch_200.h5\failed\missing\label_studio_masks')
    output_base_dir =Path(r'E:\CurrentTrainingData20240812_trn_val\Incoming_1B_Loetstift\Incoming_1B_Loetstift_unzip\main_im2_cropped_masks\time_11_04_21_val_frGrnd0.9498_epoch_200.h5\failed\missing\patches')


In [ ]:
#| eval: false
batch_extract_pins(
    image_dir=image_dir,
    mask_dir=mask_dir,
    output_base_dir=output_base_dir,
    patch_size=(87, 70),
    parallel=True
)

In [ ]:
#| eval: false
#path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/Fail18')
#tmp_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/templates')
#sn_img_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/sn_img')
#sn_img_path.mkdir(exist_ok=True, parents=True)
#path.ls()

In [ ]:
recipe_name = '18'
img = read_img(path.ls()[0], gray=True, cv=True)
tmp_img = read_img(f'{tmp_path}/{recipe_name}.png', gray=True, cv=True)
img.shape, tmp_img.shape

In [ ]:
#| eval: false
j_mPath = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/Fail18')
m_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/Fail_18_folder/jan_masks')
im_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/Fail_18_folder/jan_images')
im_src_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/Fail18')

masks = m_path.ls()
names = [Path(i).name for i in masks]
for i in tqdm(names, total=len(names)):
    shutil.copyfile(f'{im_src_path}/{i}', f'{im_path}/{i}')

In [ ]:

len(list(set([Path(i).name for i in im_path.ls()] )))

In [ ]:
len(list(set([Path(i).name for i in m_path.ls()] )))

In [ ]:
#| export
def desired_im( 
              im_path:Union[str,Path],
              tmp_path:Union[str,Path],
              msk_path:Union[str, Path,None],
              img_sn_pin_save_path:Union[str, None]=None,
              msk_sn_pin_save_path:Union[str, None]=None,
              recipe_name:str='91',
                ):
    ''

    # reading all the images
    img = read_img(im_path, cv=True, gray=True) 
    tmp_img = read_img(tmp_path, cv=True, gray=True)
    if msk_path is not None:
        msk_img = read_img(msk_path, cv=True, gray=True)
    

    name_ = Path(im_path).name

    bbox_func = globals()[f'get_all_columns_{recipe_name}']


    col_bbox = bbox_func(
                                img,
                                tmp_img
                                )

    last_part, img_part = get_full_image(
        img=img,
        im_name=name_,
        save_path=img_sn_pin_save_path,
        col_bbox=col_bbox,
        recipe_no=recipe_name
        )
    if msk_path is not None:
        last_part, mask_part = get_full_image(
                                img=msk_img,
                                im_name=name_,
                                col_bbox=col_bbox,
                                recipe_no=recipe_name,
                                save_path=msk_sn_pin_save_path,
                                )
        return last_part, mask_part
    else:
        return last_part, img_part
    

   

In [ ]:
#| export
def extract_pins(
        im_path:str, #folder where all the images are available 
        msk_path:str, # mask folder where all the masks are available
        tmp_path:str, # path to the template only path, here one file named recipe_name.png should be there
        recipe_name:str, # name of the recipe
        img_sn_pin_save_path:str, # path where the images with pins will be saved
        img_sn_pin_msk_save_path:str, # path where the images with pins and masks will be saved
        ):

    Path(img_sn_pin_save_path).mkdir(exist_ok=True, parents=True)
    tmp_path=f'{tmp_path}/{recipe_name}.png'
    if img_sn_pin_msk_save_path is not None:
        Path(img_sn_pin_msk_save_path).mkdir(exist_ok=True, parents=True)
    for i in tqdm(Path(im_path).ls(), total=len(Path(im_path).ls())):
        if msk_path is not None:
            msk_name=f'{msk_path}/{Path(i).name}'
        else:
            msk_name=None

        desired_im(
            im_path=i,
            recipe_name=recipe_name,
            msk_path=msk_name,
            tmp_path=tmp_path,
            img_sn_pin_save_path=img_sn_pin_save_path,
            msk_sn_pin_save_path=img_sn_pin_msk_save_path,
        )

In [ ]:
#| export
def get_common_files(src, dst):
    src_files = get_name_(Path(src).ls())
    dst_files = get_name_(Path(dst).ls())
    return set(src_files).intersection(set(dst_files))
    

In [ ]:
src =Path(r'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/Incoming_1B_Loetstift/Incoming_1B_Loetstift_unzip/main_im2_cropped_masks/time_11_04_21_val_frGrnd0.9498_epoch_200.h5/failed/missing/gen_im_full_from_patches/')
dst = Path(src.parent, 'gen_msk_full_from_patches')

In [ ]:
commn_files = get_common_files(src, dst)

In [ ]:
un_common_files = list(filter(lambda x: Path(x).name not in commn_files, dst.ls()))

In [ ]:
len(un_common_files)

In [ ]:
show_(only_pin_img)

In [ ]:
import nbdev; nbdev.nbdev_export()